In [1]:
pip install azure-identity azure-ai-ml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 90.6 MB/s  0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 2.0.1
    Uninstalling wrapt-2.0.1:
      Successfully uninstalled wrapt-2.0.1
  Attempting uninstall: opentelemetry-api━━━━━━━━━━━━━━━━━━━━━━━━━  6/33 [asgiref]
    Found existing installation: opentelemetry-api 1.39.1━━━━━  6/33 [asgiref]
    Uninstalling opentelemetry-api-1.39.1:━━━━━━━━━━━━━━━━━━━━━━━━  8/33 [opentelemetry-api]
      Successfully uninstalled opentelemetry-api-1.39.1━━━━━━━  8/33 [opentelemetry-api]
  Attempting uninstall: opentelemetry-semantic-conventions━━━━━━━━  8/33 [opentelemetry-api]
    Found existing installation: opentelemetry-semantic-conventions 0.60b1/33 [opentelemetry-api]
    Uninstalling opentelemetry-semantic-conventions-0.60b1:━━━  8/33 [opentelemetry-api]
      Successfully uninstalled opentelemetry-semantic-conventions-0.60b1 8/33 [opentelemetry-api]
  Attempting uninstall: azure-storage-blob━━━━━━━━━━━━

In [2]:
import glob
from pathlib import Path
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Model, ManagedOnlineDeployment
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Environment,ManagedOnlineEndpoint

In [3]:
pip show azure-ai-ml

Name: azure-ai-ml
Version: 1.32.0
Summary: Microsoft Azure Machine Learning Client Library for Python
Home-page: https://github.com/Azure/azure-sdk-for-python
Author: Microsoft Corporation
Author-email: azuresdkengsysadmins@microsoft.com
License: MIT License
Location: /anaconda/envs/jupyter_env/lib/python3.10/site-packages
Requires: azure-common, azure-core, azure-mgmt-core, azure-monitor-opentelemetry, azure-storage-blob, azure-storage-file-datalake, azure-storage-file-share, colorama, isodate, jsonschema, marshmallow, pydash, pyjwt, pyyaml, strictyaml, tqdm, typing-extensions
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [4]:
# connect to workspace
try:
    credential = DefaultAzureCredential()
    credential.get_token('https://Management.azure.com/.default')
except Exception as ex:
    credential = InteractiveBrowserCredential()

#get a handle to workspace
ml_client = MLClient.from_config(credential=credential)

Found the config file in: /config.json
Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [41]:
#Constructing project folder path
workspace_mount = Path(glob.glob('/mnt/batch/tasks/shared/LS_root/mounts/*')[0])
project_folder = list(workspace_mount.rglob("mlflow-model"))[0]
conda_path = project_folder  / 'conda.yaml'
sample_data = project_folder  / 'sample-data.json'
print(sample_data)

/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute8/code/Users/akbar.khan160659/mlflow-model/sample-data.json


In [6]:
# create an online endpoint
endpoint = ManagedOnlineEndpoint(
    name='endpoint',
    description='Online endpoint',
    auth_mode='key',
)

ml_client.begin_create_or_update(endpoint).result()





ManagedOnlineEndpoint({'public_network_access': 'Enabled', 'provisioning_state': 'Succeeded', 'scoring_uri': 'https://endpoint.uksouth.inference.ml.azure.com/score', 'openapi_uri': 'https://endpoint.uksouth.inference.ml.azure.com/swagger.json', 'name': 'endpoint', 'description': 'Online endpoint', 'tags': {}, 'properties': {'createdBy': 'Mohammad Akbar Khan', 'createdAt': '2026-03-30T11:38:18.899144+0000', 'lastModifiedAt': '2026-03-30T11:38:18.899144+0000', 'azureml.onlineendpointid': '/subscriptions/e4b55002-5402-4eb7-b4ec-feb75f3b92ac/resourcegroups/rg-bank-loan-x2rvfq/providers/microsoft.machinelearningservices/workspaces/ws-bank-loan-x2rvfq/onlineendpoints/endpoint', 'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/e4b55002-5402-4eb7-b4ec-feb75f3b92ac/providers/Microsoft.MachineLearningServices/locations/uksouth/mfeOperationsStatus/oeidp:8d9cfe1f-ddee-45b1-8469-976089cd34ab:b691928b-f07e-4c01-9e47-01ee496e8ae1?api-version=2022-02-01-preview'}, 'print_as_yaml':

In [7]:
# create env 
env = Environment(
    conda_file= conda_path,
    name='deployment-environment',
    description='Environment created from a Docker image plus Conda environment.',
    image='mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest',
)

ml_client.environments.create_or_update(env)

Environment({'arm_type': 'environment_version', 'latest_version': None, 'image': 'mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest', 'intellectual_property': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'deployment-environment', 'description': 'Environment created from a Docker image plus Conda environment.', 'tags': {}, 'properties': {'azureml.labels': 'latest'}, 'print_as_yaml': False, 'id': '/subscriptions/e4b55002-5402-4eb7-b4ec-feb75f3b92ac/resourceGroups/rg-bank-loan-x2rvfq/providers/Microsoft.MachineLearningServices/workspaces/ws-bank-loan-x2rvfq/environments/deployment-environment/versions/2', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute8/code/Users/akbar.khan160659', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x77ffe579a020>, 'serialize': <msrest.serialization.Serializer object at 0x77ffe5799660>, 'version': '2', 'conda_file': {'

In [15]:

# configure deployment
model = Model(
    path=project_folder,
    type=AssetTypes.MLFLOW_MODEL,
    description='Voting ensemble classification model',
)
blue_deployment = ManagedOnlineDeployment(
    name='blue',
    endpoint_name='endpoint',
    model=model,
    instance_type='Standard_DS3_v2',
    instance_count=1,
)


In [16]:
# create configuration
ml_client.online_deployments.begin_create_or_update(blue_deployment).result()
print('model deployed')


endpoint.traffic = {"blue": 100}
ml_client.begin_create_or_update(endpoint).result()

Check: endpoint endpoint exists


.........................................................model deployed


ManagedOnlineEndpoint({'public_network_access': 'Enabled', 'provisioning_state': 'Succeeded', 'scoring_uri': 'https://endpoint.uksouth.inference.ml.azure.com/score', 'openapi_uri': 'https://endpoint.uksouth.inference.ml.azure.com/swagger.json', 'name': 'endpoint', 'description': 'Online endpoint', 'tags': {}, 'properties': {'createdBy': 'Mohammad Akbar Khan', 'createdAt': '2026-03-30T11:38:18.899144+0000', 'lastModifiedAt': '2026-03-30T13:15:35.387966+0000', 'azureml.onlineendpointid': '/subscriptions/e4b55002-5402-4eb7-b4ec-feb75f3b92ac/resourcegroups/rg-bank-loan-x2rvfq/providers/microsoft.machinelearningservices/workspaces/ws-bank-loan-x2rvfq/onlineendpoints/endpoint', 'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/e4b55002-5402-4eb7-b4ec-feb75f3b92ac/providers/Microsoft.MachineLearningServices/locations/uksouth/mfeOperationsStatus/oeidp:8d9cfe1f-ddee-45b1-8469-976089cd34ab:2b1de996-3168-4100-b3b5-74542a782318?api-version=2022-02-01-preview'}, 'print_as_yaml':

In [46]:
# test the blue deployment with some sample data
response = ml_client.online_endpoints.invoke(
    endpoint_name='endpoint',
    deployment_name='blue',
    request_file= sample_data,
)
print(response)

[true]


In [51]:
# Get the details for online endpoint
endpoint = ml_client.online_endpoints.get(name='endpoint')
print(endpoint)

auth_mode: key
description: Online endpoint
id: /subscriptions/e4b55002-5402-4eb7-b4ec-feb75f3b92ac/resourceGroups/rg-bank-loan-x2rvfq/providers/Microsoft.MachineLearningServices/workspaces/ws-bank-loan-x2rvfq/onlineEndpoints/endpoint
identity:
  principal_id: 6f5127d6-32e8-483e-a0bf-2a6467c126c5
  tenant_id: a797f06e-03d5-4652-a858-51275caef4d4
  type: system_assigned
kind: Managed
location: uksouth
mirror_traffic: {}
name: endpoint
openapi_uri: https://endpoint.uksouth.inference.ml.azure.com/swagger.json
properties:
  AzureAsyncOperationUri: https://management.azure.com/subscriptions/e4b55002-5402-4eb7-b4ec-feb75f3b92ac/providers/Microsoft.MachineLearningServices/locations/uksouth/mfeOperationsStatus/oeidp:8d9cfe1f-ddee-45b1-8469-976089cd34ab:2b1de996-3168-4100-b3b5-74542a782318?api-version=2022-02-01-preview
  azureml.onlineendpointid: /subscriptions/e4b55002-5402-4eb7-b4ec-feb75f3b92ac/resourcegroups/rg-bank-loan-x2rvfq/providers/microsoft.machinelearningservices/workspaces/ws-bank